# Wall Image Processing — Door & Window Detection

**Input:** binary wall images from WallSegmentation (`wall_XX.png`,
black = wall, white = void, at `flat_pixel_m` resolution).

**Pipeline:**

1. Load each wall image and its metadata (pixel size in metres).
2. Find candidate void regions via connected-component analysis.
3. Use SAM (Segment Anything Model) with point prompts on each void
   to produce refined masks that bridge scan gaps.
4. Fit bounding boxes, then classify each mask as **door** or **window**
   using physical-size and position heuristics:
   - Doors touch the floor (bottom of image), are taller than wide,
     and have typical residential dimensions (~0.7–1.2 m wide, ~1.8–2.4 m tall).
   - Windows do **not** touch the floor and float in the upper portion
     of the wall.
5. Export annotated images and a structured summary.

## 0 · Environment setup

In [ ]:
!pip install -q opencv-python-headless segment-anything torch torchvision

### Download SAM checkpoint (ViT-B — smallest, fast enough for binary wall images)

In [ ]:
import os

SAM_CHECKPOINT = "sam_vit_b_01ec64.pth"
SAM_URL = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"

if not os.path.exists(SAM_CHECKPOINT):
    print("Downloading SAM ViT-B checkpoint …")
    !wget -q {SAM_URL}
    print("Done.")
else:
    print(f"Checkpoint already present: {SAM_CHECKPOINT}")

### Mount Google Drive (if running on Colab)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Imports

In [ ]:
import glob
import json
import math
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
from PIL import Image

from segment_anything import sam_model_registry, SamPredictor

## 1 · Configuration

In [ ]:
@dataclass
class Config:
    # ---- input ----
    # Point this at your Google Drive folder containing wall images.
    # Supports two layouts:
    #   flat:   wall_image_dir/wall_01.png, wall_02.png, …
    #   nested: wall_image_dir/room_01/wall_01.png, room_02/wall_01.png, …
    wall_image_dir: str = '/content/drive/MyDrive/ONESTRUCTION/wall_images'
    pixel_m: float = 0.04                    # metres per pixel (from WallSegmentation)

    # ---- SAM ----
    sam_checkpoint: str = 'sam_vit_b_01ec64.pth'
    sam_model_type: str = 'vit_b'
    sam_upscale: int = 8                     # upscale factor before feeding to SAM
    sam_points_per_void: int = 5             # prompt points sampled inside each void

    # ---- connected-component pre-filter ----
    min_void_px: int = 15                    # ignore void blobs smaller than this (noise)

    # ---- door heuristics (in metres) ----
    door_min_width_m: float = 0.50
    door_max_width_m: float = 1.60
    door_min_height_m: float = 1.50
    door_max_height_m: float = 2.80
    door_floor_margin_px: int = 3            # must touch bottom N rows to count as "on floor"

    # ---- window heuristics (in metres) ----
    window_min_width_m: float = 0.30
    window_max_width_m: float = 2.50
    window_min_height_m: float = 0.30
    window_max_height_m: float = 2.00
    window_min_sill_m: float = 0.30          # sill must be at least this far above floor

    # ---- output ----
    out_dir: str = '/content/drive/MyDrive/ONESTRUCTION/wall_openings_out'


CFG = Config()
print(f"Wall images dir : {CFG.wall_image_dir}")
print(f"Output dir      : {CFG.out_dir}")
print(f"Pixel size      : {CFG.pixel_m} m/px")
print(f"SAM upscale     : {CFG.sam_upscale}x")
print(f"Door size range : {CFG.door_min_width_m}–{CFG.door_max_width_m} m × "
      f"{CFG.door_min_height_m}–{CFG.door_max_height_m} m")
print(f"Window size     : {CFG.window_min_width_m}–{CFG.window_max_width_m} m × "
      f"{CFG.window_min_height_m}–{CFG.window_max_height_m} m")
print(f"Window min sill : {CFG.window_min_sill_m} m above floor")

## 2 · Definitions

### Load SAM model

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

sam = sam_model_registry[CFG.sam_model_type](checkpoint=CFG.sam_checkpoint)
sam.to(device)
predictor = SamPredictor(sam)
print("SAM model loaded.")

### Core functions

#### Step 1 — Find candidate voids via connected components

The binary wall image has white (255) = void. We find connected
components in the void region, filtering out noise blobs. Each
surviving component is a candidate opening (door, window, or gap).

In [ ]:
def find_void_components(wall_img, min_void_px=15):
    """Find connected void regions in a binary wall image.

    Args:
        wall_img: uint8 array, 0=wall, 255=void.
        min_void_px: ignore components smaller than this.

    Returns list of dicts with keys:
        'mask'  — bool array same size as wall_img
        'area'  — pixel count
        'bbox'  — (x, y, w, h) bounding box in image coords
        'centroid' — (cx, cy) in image coords
    """
    void_mask = (wall_img == 255).astype(np.uint8)
    n_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        void_mask, connectivity=8)

    components = []
    for i in range(1, n_labels):  # skip background (label 0)
        area = stats[i, cv2.CC_STAT_AREA]
        if area < min_void_px:
            continue

        mask = labels == i
        x = stats[i, cv2.CC_STAT_LEFT]
        y = stats[i, cv2.CC_STAT_TOP]
        w = stats[i, cv2.CC_STAT_WIDTH]
        h = stats[i, cv2.CC_STAT_HEIGHT]
        cx, cy = centroids[i]

        components.append({
            'mask': mask,
            'area': area,
            'bbox': (x, y, w, h),
            'centroid': (cx, cy),
        })

    return components

#### Step 2 — Refine voids with SAM

Each candidate void is used to generate point prompts for SAM.
SAM sees the upscaled wall image and produces a refined mask that
can bridge scan gaps and regularise ragged edges.

In [ ]:
def _sample_points_in_mask(mask, n_points=5, rng=None):
    """Sample n_points uniformly from True pixels in mask."""
    if rng is None:
        rng = np.random.default_rng(42)
    ys, xs = np.where(mask)
    if len(ys) == 0:
        return np.empty((0, 2))
    n = min(n_points, len(ys))
    idx = rng.choice(len(ys), n, replace=False)
    return np.stack([xs[idx], ys[idx]], axis=1)  # (N, 2) in x,y order


def prepare_sam_image(wall_img, upscale=8):
    """Convert binary wall image to 3-channel RGB and upscale for SAM.

    Wall (0) → dark grey, void (255) → white. The slight contrast
    helps SAM's encoder distinguish the two regions.
    """
    rgb = np.stack([wall_img, wall_img, wall_img], axis=-1)
    rgb[wall_img == 0] = 40  # dark grey for wall
    h, w = rgb.shape[:2]
    rgb_up = cv2.resize(rgb, (w * upscale, h * upscale),
                        interpolation=cv2.INTER_NEAREST)
    return rgb_up


def refine_void_with_sam(predictor, wall_img_rgb_up, component,
                         upscale=8, n_points=5):
    """Use SAM to refine a single void component.

    Returns the refined mask downscaled back to original resolution,
    and the SAM confidence score.
    """
    mask = component['mask']
    orig_h, orig_w = mask.shape

    # sample prompt points (in upscaled coords)
    pts_orig = _sample_points_in_mask(mask, n_points=n_points)
    if len(pts_orig) == 0:
        return mask, 0.0
    pts_up = pts_orig * upscale + upscale // 2  # centre of upscaled pixel
    labels = np.ones(len(pts_up), dtype=np.int32)  # all foreground

    # also add a negative point in the wall region near the void
    cx, cy = component['centroid']
    neg_candidates = []
    for dy, dx in [(-5, 0), (5, 0), (0, -5), (0, 5)]:
        ny, nx = int(cy + dy), int(cx + dx)
        if 0 <= ny < orig_h and 0 <= nx < orig_w and not mask[ny, nx]:
            neg_candidates.append((nx, ny))
    if neg_candidates:
        neg_pt = np.array([neg_candidates[0]]) * upscale + upscale // 2
        pts_up = np.vstack([pts_up, neg_pt])
        labels = np.append(labels, 0)

    predictor.set_image(wall_img_rgb_up)
    masks, scores, _ = predictor.predict(
        point_coords=pts_up,
        point_labels=labels,
        multimask_output=True,
    )

    # pick highest-scoring mask
    best_idx = np.argmax(scores)
    sam_mask_up = masks[best_idx]
    score = float(scores[best_idx])

    # downscale back to original resolution
    sam_mask_down = cv2.resize(
        sam_mask_up.astype(np.uint8),
        (orig_w, orig_h),
        interpolation=cv2.INTER_NEAREST
    ).astype(bool)

    return sam_mask_down, score

#### Step 3 — Merge nearby fragments and fit rectangular openings

Scan gaps can split a single door/window into multiple fragments.
We merge SAM masks whose bounding boxes overlap or are close together,
then fit a tight bounding rectangle to each merged region.

In [ ]:
def _bbox_overlap_or_close(b1, b2, margin=3):
    """Check if two bounding boxes overlap or are within margin px."""
    x1, y1, w1, h1 = b1
    x2, y2, w2, h2 = b2
    return not (x1 + w1 + margin < x2 or x2 + w2 + margin < x1 or
                y1 + h1 + margin < y2 or y2 + h2 + margin < y1)


def merge_fragments(refined_components, merge_margin_px=3):
    """Merge overlapping/adjacent refined masks into unified openings.

    Uses a simple union-find on bounding-box proximity.
    """
    n = len(refined_components)
    if n == 0:
        return []

    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        a, b = find(a), find(b)
        if a != b:
            parent[b] = a

    bboxes = [c['bbox'] for c in refined_components]
    for i in range(n):
        for j in range(i + 1, n):
            if _bbox_overlap_or_close(bboxes[i], bboxes[j], merge_margin_px):
                union(i, j)

    groups = {}
    for i in range(n):
        root = find(i)
        groups.setdefault(root, []).append(i)

    merged = []
    for indices in groups.values():
        combined_mask = np.zeros_like(refined_components[0]['mask'])
        for i in indices:
            combined_mask |= refined_components[i]['mask']

        ys, xs = np.where(combined_mask)
        x_min, x_max = xs.min(), xs.max()
        y_min, y_max = ys.min(), ys.max()
        bbox = (int(x_min), int(y_min),
                int(x_max - x_min + 1), int(y_max - y_min + 1))

        best_score = max(refined_components[i]['sam_score'] for i in indices)

        merged.append({
            'mask': combined_mask,
            'bbox': bbox,
            'area': int(combined_mask.sum()),
            'sam_score': best_score,
            'n_fragments': len(indices),
        })

    return merged

#### Step 4 — Classify openings as door, window, or unknown

Heuristics:
- **Touches floor:** the opening's bounding box extends into the
  bottom `door_floor_margin_px` rows of the image → candidate door.
- **Door shape:** touches floor AND width/height fall within typical
  door dimensions → **door**.
- **Window shape:** does NOT touch floor, sill is above `window_min_sill_m`,
  width/height within window range → **window**.
- Anything else → **unknown** (possibly a niche, vent, or artefact).

In [ ]:
def classify_openings(merged_openings, img_height, pixel_m, cfg):
    """Classify each merged opening as 'door', 'window', or 'unknown'.

    The image convention is: row 0 = top of wall, row (H-1) = floor.
    """
    results = []

    for opening in merged_openings:
        x, y, w_px, h_px = opening['bbox']

        w_m = w_px * pixel_m
        h_m = h_px * pixel_m

        # bottom of bounding box in image coords (row index)
        bbox_bottom_row = y + h_px - 1
        floor_row = img_height - 1

        touches_floor = (floor_row - bbox_bottom_row) <= cfg.door_floor_margin_px

        # sill height = distance from floor to bottom of opening
        sill_m = (floor_row - bbox_bottom_row) * pixel_m

        label = 'unknown'
        reason = ''

        if touches_floor:
            if (cfg.door_min_width_m <= w_m <= cfg.door_max_width_m and
                    cfg.door_min_height_m <= h_m <= cfg.door_max_height_m):
                label = 'door'
                reason = f'touches floor, size {w_m:.2f}x{h_m:.2f}m fits door range'
            else:
                label = 'door'
                reason = (f'touches floor (assumed door despite size '
                          f'{w_m:.2f}x{h_m:.2f}m outside typical range)')
        else:
            if sill_m >= cfg.window_min_sill_m:
                if (cfg.window_min_width_m <= w_m <= cfg.window_max_width_m and
                        cfg.window_min_height_m <= h_m <= cfg.window_max_height_m):
                    label = 'window'
                    reason = (f'floating at sill={sill_m:.2f}m, '
                              f'size {w_m:.2f}x{h_m:.2f}m fits window range')
                else:
                    label = 'window'
                    reason = (f'floating at sill={sill_m:.2f}m '
                              f'(assumed window despite size '
                              f'{w_m:.2f}x{h_m:.2f}m outside typical range)')
            else:
                label = 'unknown'
                reason = (f'does not touch floor but sill={sill_m:.2f}m '
                          f'is too low for a window')

        results.append({
            **opening,
            'label': label,
            'reason': reason,
            'width_m': w_m,
            'height_m': h_m,
            'sill_m': sill_m,
            'touches_floor': touches_floor,
        })

    return results

#### Full pipeline: process one wall image end-to-end

In [ ]:
def process_wall_image(img_path, predictor, cfg):
    """Full pipeline for one wall image.

    Returns list of classified openings (each a dict with label,
    bbox, dimensions, mask, etc.), or empty list if no openings found.
    """
    wall_img = np.array(Image.open(img_path).convert('L'))
    img_h, img_w = wall_img.shape

    # step 1: find void connected components
    components = find_void_components(wall_img, min_void_px=cfg.min_void_px)
    if not components:
        return [], wall_img

    # step 2: prepare upscaled image for SAM, refine each void
    rgb_up = prepare_sam_image(wall_img, upscale=cfg.sam_upscale)

    refined = []
    for comp in components:
        sam_mask, score = refine_void_with_sam(
            predictor, rgb_up, comp,
            upscale=cfg.sam_upscale,
            n_points=cfg.sam_points_per_void,
        )

        ys, xs = np.where(sam_mask)
        if len(ys) == 0:
            continue
        bbox = (int(xs.min()), int(ys.min()),
                int(xs.max() - xs.min() + 1),
                int(ys.max() - ys.min() + 1))

        refined.append({
            'mask': sam_mask,
            'bbox': bbox,
            'area': int(sam_mask.sum()),
            'sam_score': score,
        })

    if not refined:
        return [], wall_img

    # step 3: merge nearby fragments
    merged = merge_fragments(refined, merge_margin_px=cfg.door_floor_margin_px)

    # step 4: classify
    openings = classify_openings(merged, img_h, cfg.pixel_m, cfg)

    return openings, wall_img

### Visualisation

In [ ]:
LABEL_COLORS = {
    'door': (0.2, 0.6, 1.0, 0.4),      # blue
    'window': (1.0, 0.8, 0.0, 0.4),     # yellow
    'unknown': (0.7, 0.7, 0.7, 0.3),    # grey
}
LABEL_EDGE_COLORS = {
    'door': 'dodgerblue',
    'window': 'orange',
    'unknown': 'grey',
}


def plot_annotated_wall(wall_img, openings, title='', pixel_m=0.04,
                        ax=None):
    """Draw the wall image with coloured bounding boxes for each opening."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 5))

    h, w = wall_img.shape
    extent = [0, w * pixel_m, 0, h * pixel_m]
    ax.imshow(wall_img, cmap='gray', aspect='equal', extent=extent)

    for op in openings:
        bx, by, bw, bh = op['bbox']
        label = op['label']
        color = LABEL_COLORS.get(label, LABEL_COLORS['unknown'])
        edge = LABEL_EDGE_COLORS.get(label, 'grey')

        # convert to metres, flip y for extent convention
        rx = bx * pixel_m
        ry = (h - by - bh) * pixel_m  # flip: image row 0 = top
        rw = bw * pixel_m
        rh = bh * pixel_m

        rect = mpatches.FancyBboxPatch(
            (rx, ry), rw, rh,
            boxstyle="round,pad=0.01",
            facecolor=color, edgecolor=edge, linewidth=2)
        ax.add_patch(rect)

        ax.text(rx + rw / 2, ry + rh + 0.03,
                f"{label}\n{op['width_m']:.2f}x{op['height_m']:.2f}m",
                ha='center', va='bottom', fontsize=7, fontweight='bold',
                color=edge)

    ax.set_xlabel('along wall (m)')
    ax.set_ylabel('height (m)')
    ax.set_title(title)


def plot_room_results(room_results, pixel_m=0.04, cols=2):
    """Grid of annotated wall images for one room."""
    items = [(name, img, ops) for name, img, ops in room_results if img is not None]
    n = len(items)
    if n == 0:
        print("No walls to display.")
        return

    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(8 * cols, 4 * rows),
                             squeeze=False)

    for i, (name, img, ops) in enumerate(items):
        ax = axes[i // cols][i % cols]
        n_doors = sum(1 for o in ops if o['label'] == 'door')
        n_windows = sum(1 for o in ops if o['label'] == 'window')
        plot_annotated_wall(img, ops, pixel_m=pixel_m, ax=ax,
                            title=f"{name}  ({n_doors}D, {n_windows}W)")

    for j in range(n, rows * cols):
        axes[j // cols][j % cols].set_visible(False)

    plt.tight_layout()
    plt.show()

### Export helpers

In [ ]:
def save_annotated_image(wall_img, openings, out_path, pixel_m=0.04):
    """Save an annotated wall image as PNG with bounding boxes drawn."""
    h, w = wall_img.shape
    rgb = cv2.cvtColor(wall_img, cv2.COLOR_GRAY2BGR)

    color_map = {
        'door': (255, 150, 50),     # blue in BGR
        'window': (0, 200, 255),    # yellow in BGR
        'unknown': (180, 180, 180),
    }

    for op in openings:
        bx, by, bw, bh = op['bbox']
        color = color_map.get(op['label'], color_map['unknown'])
        cv2.rectangle(rgb, (bx, by), (bx + bw - 1, by + bh - 1), color, 1)
        cv2.putText(rgb, op['label'][0].upper(),
                    (bx + 2, by + bh - 2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.35, color, 1)

    cv2.imwrite(out_path, rgb)


def opening_to_dict(op):
    """Convert an opening dict to a JSON-serialisable form (drop mask)."""
    return {
        'label': op['label'],
        'bbox_px': list(op['bbox']),
        'width_m': round(op['width_m'], 3),
        'height_m': round(op['height_m'], 3),
        'sill_m': round(op['sill_m'], 3),
        'touches_floor': op['touches_floor'],
        'sam_score': round(op['sam_score'], 4),
        'n_fragments': op.get('n_fragments', 1),
        'reason': op['reason'],
    }

## 3 · Run the pipeline

### 3.1 Discover wall images

In [ ]:
# Auto-detect layout: nested (room_*/wall_*.png) or flat (*.png)
room_dirs = sorted(glob.glob(os.path.join(CFG.wall_image_dir, 'room_*')))
room_dirs = [d for d in room_dirs if os.path.isdir(d)]

if room_dirs:
    # nested layout — room_01/wall_01.png, room_02/wall_01.png, …
    print(f"Nested layout: {len(room_dirs)} room folders in {CFG.wall_image_dir}")
    room_wall_map = {}
    for rd in room_dirs:
        pngs = sorted(glob.glob(os.path.join(rd, '*.png')))
        room_name = os.path.basename(rd)
        room_wall_map[room_name] = pngs
        print(f"  {room_name}/: {len(pngs)} wall images")
else:
    # flat layout — all PNGs directly in wall_image_dir
    all_pngs = sorted(glob.glob(os.path.join(CFG.wall_image_dir, '*.png')))
    room_name = os.path.basename(CFG.wall_image_dir)
    room_wall_map = {room_name: all_pngs}
    print(f"Flat layout: {len(all_pngs)} wall images in {CFG.wall_image_dir}")
    for p in all_pngs:
        print(f"  {os.path.basename(p)}")

### 3.2 Process all rooms

In [ ]:
os.makedirs(CFG.out_dir, exist_ok=True)

all_room_results = {}
all_summaries = {}

for room_name, wall_paths in sorted(room_wall_map.items()):
    print(f"\n{'='*60}")
    print(f"  {room_name}: {len(wall_paths)} walls")
    print(f"{'='*60}")

    room_out = os.path.join(CFG.out_dir, room_name)
    os.makedirs(room_out, exist_ok=True)

    room_results = []
    room_summary = []

    for wp in wall_paths:
        wall_name = os.path.splitext(os.path.basename(wp))[0]
        print(f"  {wall_name} ... ", end='')

        try:
            openings, wall_img = process_wall_image(wp, predictor, CFG)

            n_doors = sum(1 for o in openings if o['label'] == 'door')
            n_windows = sum(1 for o in openings if o['label'] == 'window')
            n_unknown = sum(1 for o in openings if o['label'] == 'unknown')
            print(f"{len(openings)} openings "
                  f"({n_doors}D, {n_windows}W, {n_unknown}?)")

            for op in openings:
                print(f"    {op['label']:>7s}  "
                      f"{op['width_m']:.2f}x{op['height_m']:.2f}m  "
                      f"sill={op['sill_m']:.2f}m  "
                      f"SAM={op['sam_score']:.3f}  "
                      f"({op['reason']})")

            # save annotated image
            ann_path = os.path.join(room_out, f"{wall_name}_annotated.png")
            save_annotated_image(wall_img, openings, ann_path, CFG.pixel_m)

            room_results.append((wall_name, wall_img, openings))
            room_summary.append({
                'wall': wall_name,
                'openings': [opening_to_dict(o) for o in openings],
            })

        except Exception as e:
            print(f"ERROR: {e}")
            room_results.append((wall_name, None, []))

    all_room_results[room_name] = room_results
    all_summaries[room_name] = room_summary

    # save per-room JSON summary
    json_path = os.path.join(room_out, 'openings.json')
    with open(json_path, 'w') as f:
        json.dump(room_summary, f, indent=2)

print(f"\n{'='*60}")
print("DONE")
total_doors = sum(
    sum(1 for o in ops if o['label'] == 'door')
    for results in all_room_results.values()
    for _, _, ops in results
)
total_windows = sum(
    sum(1 for o in ops if o['label'] == 'window')
    for results in all_room_results.values()
    for _, _, ops in results
)
print(f"Total doors: {total_doors}, windows: {total_windows}")
print(f"Output: {CFG.out_dir}")

### 3.3 Visualise results per room

In [ ]:
for room_name, results in sorted(all_room_results.items()):
    print(f"\n--- {room_name} ---")
    plot_room_results(results, pixel_m=CFG.pixel_m)

### 3.4 Summary table

In [ ]:
print(f"{'Room':<12} {'Wall':<12} {'Label':<10} {'Width(m)':<10} "
      f"{'Height(m)':<10} {'Sill(m)':<10} {'SAM':<8} {'Fragments':<10}")
print('-' * 92)

for room_name, summary in sorted(all_summaries.items()):
    for wall_entry in summary:
        wall = wall_entry['wall']
        for op in wall_entry['openings']:
            print(f"{room_name:<12} {wall:<12} {op['label']:<10} "
                  f"{op['width_m']:<10.2f} {op['height_m']:<10.2f} "
                  f"{op['sill_m']:<10.2f} {op['sam_score']:<8.3f} "
                  f"{op['n_fragments']:<10}")

### 3.5 Save combined JSON summary

All openings across all rooms in one file for downstream BIM conversion.

In [ ]:
combined_path = os.path.join(CFG.out_dir, 'all_openings.json')
with open(combined_path, 'w') as f:
    json.dump(all_summaries, f, indent=2)

print(f"Combined summary saved to: {combined_path}")
print(f"\nOutput directory contents:")
for root, dirs, files in os.walk(CFG.out_dir):
    level = root.replace(CFG.out_dir, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = '  ' * (level + 1)
    for f in sorted(files):
        print(f"{sub_indent}{f}")